# A first footprint

This notebook calculates the carbon footprint of one toy T-shirt, step by step. At each step it says what you type, what you get back, and what it means in LCA terms. Then it asks why the number is what it is, changes one input, and calculates again.

You will:

1. load a small background database,
2. read the score of each product it holds,
3. open a database and calculate the footprint of one T-shirt,
4. read what it is made of (`contributions`), who emits it (`where`) and which lines matter most (`sensitivity`),
5. change one amount and calculate again,
6. add a waste line.

Every number was worked out by hand in [docs/calculation.md](../docs/calculation.md#the-toy-background). The code checks the main ones with `assert`, so CI fails if a result ever moves. [00-getting-started](00-getting-started.ipynb) lists every function and type used here, and the [glossary](../docs/appendix/glossary.md) explains the terms.

## Set up

This cell imports headwater as `hw` and finds the toy files in `tests/fixtures/toy`, looking from the notebook's folder upwards, so it works wherever Jupyter starts. Look at the list of files: a background and its methods, all small text files. The other folders there hold a few toy databases, one folder per case.

In [1]:
import difflib
import math
import shutil
import tempfile
from pathlib import Path

import headwater as hw

here = Path.cwd()
toy = next(
    (
        p / "tests/fixtures/toy"
        for p in [here, *here.parents]
        if (p / "tests/fixtures/toy").is_dir()
    ),
    None,
)
assert toy, "tests/fixtures/toy not found: run this notebook from a checkout of the repository"

for path in sorted(toy.rglob("*")):
    if path.is_file() and path.relative_to(toy).parts[0] in ("background", "methods"):
        print(path.relative_to(toy))

background/flows.csv
background/processes/electricity-production.csv
background/processes/electricity-production.yaml
background/processes/fabric-production.csv
background/processes/fabric-production.yaml
background/processes/incineration-textile-waste.csv
background/processes/incineration-textile-waste.yaml
methods/toy-climate/factors.csv
methods/toy-climate/method.yaml
methods/toy-impacts/factors.csv
methods/toy-impacts/method.yaml
methods/toy-nw/factors.csv
methods/toy-nw/set.yaml


## The background

A **background** is the shared database that your own processes buy from. ecoinvent is one. Here it is three tiny processes:

| Process | Makes | Buys | Emits |
|---|---|---|---|
| electricity production | 1 kWh electricity | | 0.5 kg CO₂, 0.001 kg methane |
| fabric production | 1 kg cotton fabric | 3 kWh electricity | 2.0 kg CO₂, 0.004 kg methane |
| incineration of textile waste | treats 1 kg textile waste | | 1.2 kg CO₂ |

The **method** `toy-climate` has one indicator, climate change. Its **characterisation factors** say how much each emission counts: 1 kg of CO₂ is 1 kg CO₂-eq, and 1 kg of methane is 29.8 kg CO₂-eq.

In Brightway each process would be an activity, and each row of its file an exchange. In headwater a row is a **line**. This is the fabric process:

In [2]:
print((toy / "background/processes/fabric-production.csv").read_text())

line,role,flow,amount,unit,supplier
e01,reference,toy:cotton-fabric,1,kg,
e02,input,toy:electricity,3,kWh,toy:electricity-production
e03,emission,toy:co2,2.0,kg,
e04,emission,toy:ch4,0.004,kg,



Read it line by line. Each line has a **role**, which says what it means:

- `reference`: what the process makes, 1 kg of cotton fabric. Brightway calls it the production exchange.
- `input`: what it buys, 3 kWh of electricity, from the process named in `supplier`. Brightway calls it a technosphere exchange.
- `emission`: what it releases to nature, with no supplier. Brightway calls it a biosphere exchange.

Amounts are always positive. The role decides what the sign means in the calculation.

## Load the background

**You type** `hw.load(folder, method=..., namespace=...)`. It reads the processes and the method, checks them, builds the matrices and solves them once. `namespace="toy"` is the start of every process id: the fabric process becomes `toy:fabric-production`.

**You get** a `Background`. It holds the score of one unit of every product, for every indicator of the method, with everything upstream included. Brightway would compute those one LCA at a time; here they are all ready, so your own processes need no new solve of the background.

In [3]:
bg = hw.load(toy / "background", method=toy / "methods/toy-climate", namespace="toy")
print(bg)

Background(folder background, namespace toy, method toy-climate 1, 1 indicator, 3 processes)


## The score of each product

**You type** `bg.score(process_id)`. The id names the process that makes the product, as the background's own processes do.

**You get** a number: the score of one unit of that product, with everything upstream included. These are the scores per unit of the three products. `bg.contributions(process_id)` says what one is made of.

In [4]:
for process_id, per in [
    ("toy:electricity-production", "kWh"),
    ("toy:fabric-production", "kg"),
    ("toy:incineration-textile-waste", "kg treated"),
]:
    print(f"{process_id:32} score {bg.score(process_id):.5f} kg CO2-eq per {per}")

assert math.isclose(bg.score("toy:fabric-production"), 3.7086, rel_tol=1e-9)

toy:electricity-production       score 0.52980 kg CO2-eq per kWh
toy:fabric-production            score 3.70860 kg CO2-eq per kg
toy:incineration-textile-waste   score 1.20000 kg CO2-eq per kg treated


**It means.** Check the fabric by hand: 2.0 kg CO₂ × 1, plus 0.004 kg methane × 29.8, plus 3 kWh × 0.5298 = 2.0 + 0.1192 + 1.5894 = 3.7086. The last term is the electricity the fabric buys, at electricity's own score. That is why the background is solved once: every score already holds the scores upstream.

The incineration score is positive: 1.2 per kg treated. Sending waste to treatment adds a burden. Inside the matrix headwater uses ecoinvent's sign convention, where a treatment's reference is -1 kg, but it always reports a treatment per kg treated, as a positive number.

Print its contributions to see the fabric's score as a tree:

In [5]:
print(bg.contributions("toy:fabric-production"))

1 kg of cotton fabric production: 3.70860 (100.0%), direct 2.11920
  3 kWh of electricity production: 1.58940 (42.9%), direct 1.58940


## Open the database

A **database** is a folder of the processes you model yourself, Brightway's foreground. Here it holds one T-shirt. It is a folder of small text files, so it can live in git, and the T-shirt is one process with its own pair of files. This is what one T-shirt buys:

In [6]:
database_folder = toy / "tshirt/tshirt"
print((database_folder / "processes/tshirt.csv").read_text())

line,role,flow,amount,unit,supplier
e01,reference,tshirt:tshirt,1,unit,
e02,input,toy:cotton-fabric,0.25,kg,toy:fabric-production
e03,input,toy:electricity,0.5,kWh,toy:electricity-production



One T-shirt (line `e01`, `reference`) buys 0.25 kg of cotton fabric (`e02`) and 0.5 kWh of electricity (`e03`). The `supplier` column names the background process that makes each input, by its id.

`database.yaml` holds the rest: the database's `name`, which starts the id of each of its processes, the `methods` it is scored with, and `background`, the release it was written for. This database names `toy-background@fixture`, a label for the toy folder. Every process of a database is a product, scored per unit of what it makes: the T-shirt per unit. A study, a folder next to the database, asks it a question: which products, and how much of each ([notebook 02](02-writing-a-database.ipynb)).

In [7]:
print((database_folder / "database.yaml").read_text())

name: tshirt
background: toy-background@fixture
methods: [toy-climate]



## Calculate

**You type** `hw.open(folder)`, which reads and checks the files and gives a `Database`, then `db.calc(bg, "tshirt:tshirt")`, with the id of the process to score.

**You get** a `Result`: the score of one unit of what that process makes, one T-shirt, for the method's indicator. Printing it shows the process, the score with its unit, the indicator and the method.

In [8]:
db = hw.open(database_folder)
result = db.calc(bg, "tshirt:tshirt")
print(result)
assert math.isclose(result.score, 1.19205, rel_tol=1e-9)

Result(tshirt:tshirt = 1.19205 kg CO2-eq, indicator climate-change, method toy-climate 1)


**It means.** One T-shirt is 1.19205 kg CO₂-eq. By hand: 0.25 kg × 3.7086 + 0.5 kWh × 0.5298 = 0.92715 + 0.2649 = 1.19205. The background was not solved again: the T-shirt's purchases were multiplied by the stored scores.

The number is in `result.score`, and its unit in `result.unit`:

In [9]:
print(f"{result.score:.5f} {result.unit}")
print(f"indicator {result.indicator}, for one unit of what process {result.process_id} makes")

1.19205 kg CO2-eq
indicator climate-change, for one unit of what process tshirt:tshirt makes


## Two roads to the same number

`result.score` adds up what the T-shirt buys, each at its stored score. That is the backward road. `result.forward_score` takes the forward road, the one Brightway's `lci()` takes: it works out how much every process has to run for one T-shirt, the supply, then adds up their direct emissions. The two must agree.

In [10]:
print(f"score         {result.score:.5f}")
print(f"forward_score {result.forward_score:.5f}")
assert math.isclose(result.score, result.forward_score, rel_tol=1e-9)

score         1.19205
forward_score 1.19205


## What the result remembers

A result carries its **provenance**: what it was computed from, and how. It is in `result.provenance`. Here are a few fields. The full record also holds a hash of the inputs, the engine version, the solver settings and, for a release, its name and hash.

In [11]:
p = result.provenance
print("method:    ", p.method, p.method_version)
print("indicator: ", p.indicator)
print("pin:       ", p.pin)
print("residual below tolerance:", p.residual <= p.solver.tolerance)

method:     toy-climate 1
indicator:  climate-change
pin:        toy-background@fixture
residual below tolerance: True


The **residual** says how far the computed answer is from satisfying the equations: it is the accuracy check. headwater raises an error rather than return a number that fails it.

## Why is the number 1.19205?

**You type** `result.contributions()`.

**You get** the same kind of tree as for the fabric: each node is a process, with its score, its share of the T-shirt, and its direct emissions. A node's score is its direct emissions, plus the score of each purchase shown under it, plus a `rest` for purchases too small to show. A purchase is the amount bought × the stored score of what is bought.

In [12]:
root = result.contributions()
print(root)

fabric, electricity = root.children
assert math.isclose(fabric.score, 0.92715, rel_tol=1e-9)
assert math.isclose(electricity.score, 0.2649, rel_tol=1e-9)

1 unit of T-shirt: 1.19205 (100.0%), direct 0.00000
  0.25 kg of cotton fabric production: 0.92715 (77.8%), direct 0.52980
    0.75 kWh of electricity production: 0.39735 (33.3%), direct 0.39735
  0.5 kWh of electricity production: 0.26490 (22.2%), direct 0.26490


**It means.** Fabric is 77.8% of the footprint, and the electricity the T-shirt buys directly is 22.2%. Fabric production buys electricity too: 0.75 kWh for 0.25 kg of fabric, worth 0.39735, shown under the fabric.

Small purchases are folded into `rest`, so a big tree stays readable. By default a purchase under 3% of the score is folded (`min_share=0.03`). Raise it to 50% to see it happen. Every node still adds up: direct + children + rest = score.

In [13]:
short = result.contributions(min_share=0.5)
print(short)
total = short.direct + sum(child.score for child in short.children) + short.rest
assert math.isclose(total, short.score, rel_tol=1e-9)

1 unit of T-shirt: 1.19205 (100.0%), direct 0.00000
  0.25 kg of cotton fabric production: 0.92715 (77.8%), direct 0.52980
    rest: 0.39735 (33.3%)
  rest: 0.26490 (22.2%)


## Who emits it?

`contributions` follows what the T-shirt **buys**. `result.where()` answers another question: which processes **emit**, wherever they are upstream, and in which locations. A process's part is its direct emissions times how much it runs for one T-shirt. This is what Brightway users call a contribution analysis by process.

In [14]:
print(result.where())

1 unit of T-shirt: 1.19205
By process:
  1.25 kWh of electricity production (GLO): 0.66225 (55.6%)
  0.25 kg of cotton fabric production (GLO): 0.52980 (44.4%)
By location:
  GLO: 1.19205 (100.0%), 2 processes


**It means.** Electricity production runs 1.25 kWh for one T-shirt: 0.5 bought directly and 0.75 through the fabric. So it emits 1.25 × 0.5298 = 0.66225, more than half the footprint, although the T-shirt buys only 0.26490 of electricity directly. On ecoinvent the two views differ much more.

## Which lines matter most?

A **sensitivity analysis** ranks every line in the files that someone could change: an amount bought, an emission, a waste sent. `result.sensitivity()` ranks them all, in the database and in the background it reaches. Two numbers say how much each matters:

- **sensitivity**: how much the footprint changes if the line's amount goes up by one unit,
- **elasticity**: by how many % the footprint changes if the amount goes up by 1%.

**You get** a tuple of `Sensitivity` objects, largest elasticity first. Here are the top five, as a table.

In [15]:
top = result.sensitivity(top=5)
print(f"{'process':28} {'line':4} {'role':9} {'amount':>12} {'sensitivity':>12} {'elasticity':>10}")
for line in top:
    amount = f"{line.amount:g} {line.unit}"
    print(
        f"{line.process_id:28} {line.line_id:4} {line.role.value:9} {amount:>12} "
        f"{line.sensitivity:12.5f} {line.elasticity:10.5f}"
    )
by_line = {(line.process_id, line.line_id): line for line in top}
assert math.isclose(by_line["toy:fabric-production", "e02"].sensitivity, 0.13245, rel_tol=1e-9)

process                      line role            amount  sensitivity elasticity
tshirt:tshirt                e02  input          0.25 kg      3.70860    0.77778
toy:electricity-production   e02  emission        0.5 kg      1.25000    0.52431
toy:fabric-production        e03  emission          2 kg      0.25000    0.41945
toy:fabric-production        e02  input            3 kWh      0.13245    0.33333
tshirt:tshirt                e03  input          0.5 kWh      0.52980    0.22222


**It means.** The first row is the T-shirt's own line `e02`, the 0.25 kg of fabric. Its elasticity is 0.7778: 1% more fabric means 0.78% more footprint. Its sensitivity is 3.7086, the score of one kg of fabric.

The fourth row is deeper, in the background: the 3 kWh of electricity that each kg of fabric needs. One more kWh per kg of fabric adds 0.13245 to the T-shirt. That is the score of electricity (0.5298) times the 0.25 kg of fabric in the T-shirt.

The elasticities of all emissions add up to 1. The score is linear in the emissions: make every emission 1% bigger and the footprint grows 1%.

In [16]:
emissions = sum(line.elasticity for line in result.sensitivity() if line.role.value == "emission")
print(f"{emissions:.5f}")
assert math.isclose(emissions, 1.0, rel_tol=1e-9)

1.00000


## Change one amount

A database is text, so a change is an edit of one line. This cell copies the folder to a temporary place, changes 0.25 kg of fabric to 0.30 kg, and prints the difference, the way git shows it. The toy files stay as they are.

In [17]:
workdir = tempfile.TemporaryDirectory()
edited = Path(workdir.name) / "tshirt-more-fabric/tshirt"
shutil.copytree(database_folder, edited)

csv = edited / "processes/tshirt.csv"
csv.write_text(csv.read_text().replace("0.25", "0.30"))


def show_diff(before, after, before_name, after_name):
    print(
        "\n".join(
            difflib.unified_diff(
                before.read_text().splitlines(),
                after.read_text().splitlines(),
                before_name,
                after_name,
                lineterm="",
                n=0,
            )
        )
    )


show_diff(database_folder / "processes/tshirt.csv", csv, "tshirt", "tshirt-more-fabric")

--- tshirt
+++ tshirt-more-fabric
@@ -3 +3 @@
-e02,input,toy:cotton-fabric,0.25,kg,toy:fabric-production
+e02,input,toy:cotton-fabric,0.30,kg,toy:fabric-production


Before calculating, the sensitivity predicts the answer. The fabric line has a sensitivity of 3.7086, and we added 0.05 kg, so the footprint should grow by 3.7086 × 0.05 = 0.18543. Now calculate the edited database. The background is not solved again.

In [18]:
fabric_line = by_line["tshirt:tshirt", "e02"]
predicted = result.score + fabric_line.sensitivity * (0.30 - 0.25)

more_fabric = hw.open(edited).calc(bg, "tshirt:tshirt")
print(f"predicted from the sensitivity: {predicted:.5f}")
print(f"calculated:               {more_fabric.score:.5f}")
assert math.isclose(more_fabric.score, 1.37748, rel_tol=1e-9)
assert math.isclose(more_fabric.score, predicted, rel_tol=1e-9)

predicted from the sensitivity: 1.37748
calculated:               1.37748


The prediction is exact here because the score is linear in the amount the T-shirt buys. For a line deeper in a loop it is a first estimate.

In a real repository you would commit the change, and `headwater diff` would show the scores before and after, with line `e02` as the cause ([00-getting-started](00-getting-started.ipynb) runs it).

## The waste variant

Real products make scraps. The toy database in `tshirt-waste` sends 0.05 kg of textile waste to incineration: one more line. Here is the difference between the two files.

In [19]:
waste_folder = toy / "tshirt-waste/tshirt"
show_diff(
    database_folder / "processes/tshirt.csv",
    waste_folder / "processes/tshirt.csv",
    "tshirt",
    "tshirt-waste",
)

--- tshirt
+++ tshirt-waste
@@ -4,0 +5 @@
+e04,waste_to_treatment,toy:textile-waste,0.05,kg,toy:incineration-textile-waste


The new line has the role `waste_to_treatment` and a positive amount: 0.05 kg sent. In Brightway the same line is a technosphere exchange of -0.05 kg, following ecoinvent's convention. Incineration emits 1.2 kg CO₂ per kg it treats, so 0.05 kg adds 0.05 × 1.2 = 0.06. The T-shirt should now score 1.19205 + 0.06 = 1.25205.

In [20]:
with_waste = hw.open(waste_folder).calc(bg, "tshirt:tshirt")
print(with_waste)
print(f"waste adds {with_waste.score - result.score:.5f}")
assert math.isclose(with_waste.score, 1.25205, rel_tol=1e-9)

Result(tshirt:tshirt = 1.25205 kg CO2-eq, indicator climate-change, method toy-climate 1)
waste adds 0.06000


The tree shows the new purchase as its own branch.

In [21]:
print(with_waste.contributions())

1 unit of T-shirt: 1.25205 (100.0%), direct 0.00000
  0.25 kg of cotton fabric production: 0.92715 (74.1%), direct 0.52980
    0.75 kWh of electricity production: 0.39735 (31.7%), direct 0.39735
  0.5 kWh of electricity production: 0.26490 (21.2%), direct 0.26490
  0.05 kg of incineration of textile waste: 0.06000 (4.8%), direct 0.06000


## What you did

- Loaded a background once (`hw.load`) and read the score of each product (`bg.score`).
- Opened a database (`hw.open`), calculated its T-shirt (`db.calc`), and read the provenance.
- Asked what it is made of (`contributions`), who emits it (`where`) and what moves it (`sensitivity`).
- Changed one amount, with its sensitivity predicting the answer before the calculation.
- Added a waste line and saw it count as a burden.

Next, [02-writing-a-database](02-writing-a-database.ipynb): write a database folder from scratch, file by file, and see how headwater tells you what is wrong with it.